In [0]:
import pandas as pd

dbutils.widgets.text("master_data_file", "/Volumes/workspace/hdb/hdb-output-data/hdb_master_data.csv")
dbutils.widgets.text("cleaned_data_file", "/Volumes/workspace/hdb/hdb-output-data/hdb_cleaned_data.csv")
dbutils.widgets.text("failed_data_file", "/Volumes/workspace/hdb/hdb-output-data/hdb_failed_data.csv")

master_data_file = dbutils.widgets.get("master_data_file")
cleaned_data_file = dbutils.widgets.get("cleaned_data_file")
failed_data_file = dbutils.widgets.get("failed_data_file")

df = pd.read_csv(master_data_file)

In [0]:
# flat_model - get to uppercase and remove hyphens 
df['flat_type'] = df['flat_type'].str.upper().str.replace('-',' ').str.strip()
df['flat_type'].unique()


In [0]:
# flat_model - get to uppercase and remove hyphens 
df['flat_model'] = df['flat_model'].str.upper().str.replace('-',' ').str.strip()
df['flat_model'].unique()

In [0]:
# block - remove letters if available. add preceeding zeros so block is a 3 digit number
df['block'] = df['block'].str.replace(r'[a-zA-Z]','', regex=True).str.zfill(3)
df['block'].unique()


In [0]:
# Assuming HDB lease is 99 years old, recompute remaining lease as of today. Remaining lease should be rounded down to Years and Months.
from datetime import datetime
from dateutil.relativedelta import relativedelta

today = pd.Timestamp.now()
LEASE_PERIOD = 99

def get_time_diff(start_date):
    diff = relativedelta(start_date, today)
    return f"{diff.years} years {diff.months} months"

df['lease_end_year'] = df['lease_commence_date'].astype(int) + LEASE_PERIOD
df['lease_end_month'] = pd.to_datetime(df['lease_end_year'].astype(str) + '-01-01')
df['remaining_lease'] = df['lease_end_month'].apply(get_time_diff)
df = df.drop(columns=['lease_end_year', 'lease_end_month'])
df.head(10)


In [0]:
# Get duplicate record by all the columns except 'resale_price'. Keep the highest price and discard other records.
df_sorted = df.sort_values(by='resale_price', ascending=False)
df_cleaned = df_sorted.duplicated(subset=['month', 'town', 'flat_type', 'block',
                                             'street_name', 'storey_range', 'floor_area_sqm',
                                             'flat_model', 'lease_commence_date'], keep='first')

unwanted_duplicates = df_sorted[df_cleaned].sort_index()
df_cleaned = df_sorted[~df_cleaned].sort_index()

unwanted_duplicates.to_csv(failed_data_file)
unwanted_duplicates.head(10)

    

In [0]:
# calculate resale average price by grouping into 'month', 'town', 'flat_type' for hash function 

df_cleaned['resale_price_avg'] = df_cleaned.groupby(['month', 'town', 'flat_type'])['resale_price'].transform('mean').round(0).astype(int)

df_cleaned.to_csv(cleaned_data_file)
df_cleaned.head(10)